# Chest X_Ray dataset by Paul Mooney

In [ ]:
%pip install -q kaggle

In [ ]:
import os
import sys

def setup_kaggle():
    if 'google.colab' in sys.modules:
        # Colab setup
        from google.colab import files
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        !mv kaggle.json /root/.kaggle/
        !chmod 600 /root/.kaggle/kaggle.json
    else:
        # Local Jupyter setup: accepts a new API token (~/.kaggle/access_token or
        # KAGGLE_API_TOKEN) or a legacy key (~/.kaggle/kaggle.json)
        kaggle_dir = os.path.expanduser(os.path.join("~", ".kaggle"))
        found = [name for name in ("access_token", "kaggle.json")
                 if os.path.exists(os.path.join(kaggle_dir, name))]
        if os.environ.get("KAGGLE_API_TOKEN"):
            found.append("KAGGLE_API_TOKEN")
        if not found:
            raise FileNotFoundError(f"❌ No Kaggle credentials found in {kaggle_dir}")
        print("✅ Kaggle credentials found:", ", ".join(found))

setup_kaggle()

In [ ]:
!kaggle --version

In [ ]:
import os

# Works on Windows, macOS and Linux
for folder in ["raw", "interim", "processed"]:
    os.makedirs(os.path.join("data", folder), exist_ok=True)

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data/raw

In [ ]:
import os

print(os.listdir("data/raw"))

In [ ]:
import zipfile

zip_path = "data/raw/chest-xray-pneumonia.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data/interim")

print("Extracted Successfully!")

In [ ]:
import shutil
import os

# The official val/ split has only 16 images, so merge it into train/.
# src/dataset.py then makes a stratified train/validation split.
base_dir = os.path.join("data", "interim", "chest_xray")

for category in ["NORMAL", "PNEUMONIA"]:
    val_path = os.path.join(base_dir, "val", category)
    train_path = os.path.join(base_dir, "train", category)
    if not os.path.isdir(val_path):
        continue  # already merged

    for file in os.listdir(val_path):
        src = os.path.join(val_path, file)
        dst = os.path.join(train_path, file)
        shutil.move(src, dst)

print("Merged val into train")

In [ ]:
# Images per split and class
for split in ["train", "val", "test"]:
    for category in ["NORMAL", "PNEUMONIA"]:
        path = os.path.join(base_dir, split, category)
        if os.path.isdir(path):
            print(f"{split}/{category}: {len(os.listdir(path))}")